# recipe-dataclass — worked example 2: Recipe for a binary matmul_forward

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `recipe-dataclass`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A binary op records BOTH Tensor inputs in `parents`, keyed by their positional argnum: `{0: a, 1: b}`. The `args` tuple stores both raw arrays in positional order. Keeping both parents is what lets the reverse pass send a gradient down each branch of the graph; omitting one orphans that branch.

## Worked solution

`matmul_forward(a, b)` computes the raw product `a.array @ b.array` and wraps it in a fresh `MiniTensor`. We then build the `Recipe`. `func` is `t.matmul` so the dispatcher finds the matmul backward rule. `args` is `(a.array, b.array)` — both inputs unboxed, in positional order, because matmul's backward needs both raw operands to form the transpose-products. `kwargs` is empty. `parents` is `{0: a, 1: b}`: argument 0 is `a` and argument 1 is `b`, both referring to the original `MiniTensor` objects (not their arrays) because the reverse pass writes `.grad` back onto those exact objects. Contrast with the unary case, which has just `{0: x}`.

In [ ]:
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def matmul_forward(a: MiniTensor, b: MiniTensor) -> MiniTensor:
    out = MiniTensor(a.array @ b.array)
    out.recipe = Recipe(
        func=t.matmul,
        args=(a.array, b.array),
        kwargs={},
        parents={0: a, 1: b},
    )
    return out


t.manual_seed(0)
a = MiniTensor(t.randn(2, 3))
b = MiniTensor(t.randn(3, 4))
out = matmul_forward(a, b)
print('out shape:', tuple(out.array.shape))
print('parent 0 is a:', out.recipe.parents[0] is a)
print('parent 1 is b:', out.recipe.parents[1] is b)